# Amazon Week 5 V3 — Model Comparison + Ablation

Run **Runtime → Run all** in Google Colab.

Upload these three ZIPs when prompted:
1. `amazon_week5_v1_feature_dataset.zip`
2. `amazon_week5_v2_lambdamart_results.zip`
3. `amazon_week2_baseline_ranking_results*.zip`


In [ ]:
# ============================================================
# AMAZON WEEK 5 V3 — MODEL COMPARISON + ABLATION ONE-CLICK
#
# Upload these 3 ZIP files when prompted:
# 1) amazon_week5_v1_feature_dataset.zip
# 2) amazon_week5_v2_lambdamart_results.zip
# 3) amazon_week2_baseline_ranking_results*.zip
#
# Output:
# amazon_week5_v3_model_comparison_ablation_results.zip
#
# What this notebook does:
# - Formal Week2 vs Week5 model comparison
# - Full LambdaMART reproducibility check
# - Feature-group ablation study
# - Query-level XGBoost vs LambdaMART comparison
# - Paired bootstrap CI for NDCG@10 difference
# ============================================================

!pip -q install "lightgbm>=4.0,<5.0" "scikit-learn>=1.4" pandas numpy matplotlib pyarrow

import os, json, shutil, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
from sklearn.metrics import ndcg_score
from google.colab import files

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

ROOT = Path("/content/week5_v3")
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# STEP 1 — Upload the 3 ZIP files
# ------------------------------------------------------------
print("=" * 74)
print("STEP 1/8 — Upload Week5 V1, Week5 V2, and Week2 baseline ZIPs")
print("=" * 74)

uploaded = files.upload()
names = list(uploaded.keys())

def pick_zip(required_terms):
    for name in names:
        low = name.lower()
        if name.lower().endswith(".zip") and all(t.lower() in low for t in required_terms):
            return name
    return None

v1_zip = pick_zip(["week5", "v1"])
v2_zip = pick_zip(["week5", "v2"])
w2_zip = pick_zip(["week2", "baseline"])

missing = []
if v1_zip is None: missing.append("Week5 V1")
if v2_zip is None: missing.append("Week5 V2")
if w2_zip is None: missing.append("Week2 baseline")

if missing:
    print("\nUploaded:")
    for x in names:
        print(" -", x)
    raise ValueError("Could not identify: " + ", ".join(missing))

print("✓ Week5 V1:", v1_zip)
print("✓ Week5 V2:", v2_zip)
print("✓ Week2 baseline:", w2_zip)

def extract(zip_name, folder_name):
    out = ROOT / folder_name
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(out)
    return out

V1 = extract(v1_zip, "v1")
V2 = extract(v2_zip, "v2")
W2 = extract(w2_zip, "week2")

def find_file(folder, filename):
    hits = list(Path(folder).rglob(filename))
    if not hits:
        hits = [
            p for p in Path(folder).rglob("*")
            if p.is_file() and p.name.lower() == filename.lower()
        ]
    if not hits:
        raise FileNotFoundError(f"Could not find {filename} inside {folder}")
    return hits[0]

# ------------------------------------------------------------
# STEP 2 — Load all artifacts
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 2/8 — Load V1 features, V2 model outputs, Week2 baselines")
print("=" * 74)

# V1
v1_parquet = find_file(V1, "week5_feature_dataset.parquet")
v1_features_json = find_file(V1, "week5_feature_list.json")

df = pd.read_parquet(v1_parquet)
with open(v1_features_json, "r", encoding="utf-8") as f:
    all_features = json.load(f)

# V2
v2_metrics = pd.read_csv(find_file(V2, "week5_v2_test_metrics.csv"))
v2_rankings = pd.read_csv(find_file(V2, "week5_v2_test_rankings.csv"))
with open(find_file(V2, "week5_v2_config.json"), "r", encoding="utf-8") as f:
    v2_config = json.load(f)

# Week2
w2_metrics = pd.read_csv(find_file(W2, "benchmark_metrics.csv"))
xgb_pred = pd.read_csv(find_file(W2, "xgboost_predictions.csv"))

for frame in [df, v2_rankings, xgb_pred]:
    frame["query_id"] = frame["query_id"].astype(str)
    frame["product_id"] = frame["product_id"].astype(str)

df["relevance_label"] = pd.to_numeric(df["relevance_label"], errors="raise").astype(int)

train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

print(f"V1 rows        : {len(df):,}")
print(f"Features       : {len(all_features)}")
print(f"Train queries  : {train_df['query_id'].nunique()}")
print(f"Test queries   : {test_df['query_id'].nunique()}")
print(f"Week2 XGB rows : {len(xgb_pred):,}")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def sort_rank(frame):
    return frame.sort_values(
        ["query_id", "product_id"],
        kind="mergesort"
    ).reset_index(drop=True)

def group_sizes(frame):
    return (
        frame.groupby("query_id", sort=False)
        .size()
        .astype(int)
        .tolist()
    )

def mean_ndcg(frame, score_col, k=10):
    vals = []
    for _, g in frame.groupby("query_id", sort=False):
        y = g["relevance_label"].to_numpy(dtype=float)
        s = g[score_col].to_numpy(dtype=float)
        vals.append(
            ndcg_score(
                y.reshape(1, -1),
                s.reshape(1, -1),
                k=k
            )
        )
    return float(np.mean(vals))

def ranking_metrics(frame, score_col):
    ndcg10 = []
    ndcg3 = []
    recalls = []
    mrrs = []

    for _, g in frame.groupby("query_id", sort=False):
        y = g["relevance_label"].to_numpy(dtype=float)
        s = g[score_col].to_numpy(dtype=float)

        ndcg10.append(
            ndcg_score(y.reshape(1,-1), s.reshape(1,-1), k=10)
        )
        ndcg3.append(
            ndcg_score(y.reshape(1,-1), s.reshape(1,-1), k=3)
        )

        order = np.argsort(-s, kind="mergesort")
        ranked = y[order]
        top10 = ranked[:10]

        n_rel = int((y > 0).sum())
        recalls.append(
            float((top10 > 0).sum() / n_rel) if n_rel > 0 else 0.0
        )

        pos = np.where(ranked > 0)[0]
        mrrs.append(
            float(1.0 / (pos[0] + 1)) if len(pos) else 0.0
        )

    return {
        "NDCG@10": float(np.mean(ndcg10)),
        "NDCG@3": float(np.mean(ndcg3)),
        "Recall@10": float(np.mean(recalls)),
        "MRR": float(np.mean(mrrs)),
        "Queries": int(frame["query_id"].nunique())
    }

# ------------------------------------------------------------
# STEP 3 — Formal model comparison
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 3/8 — Build formal Week2 vs Week5 model comparison")
print("=" * 74)

comparison_rows = []

# Use Week2 published baseline metrics for exact historical comparison.
for _, row in w2_metrics.iterrows():
    comparison_rows.append({
        "model": row["model"],
        "stage": "Week 2",
        "NDCG@10": float(row["NDCG@10"]),
        "Recall@10": float(row["Recall@10"]),
        "MRR": float(row["MRR"]),
        "NDCG@3": np.nan
    })

# Add V2 LambdaMART and semantic signals.
for _, row in v2_metrics.iterrows():
    model = str(row["model"])
    if model in ["LambdaMART", "SBERT", "CLIP"]:
        comparison_rows.append({
            "model": model,
            "stage": "Week 5",
            "NDCG@10": float(row["NDCG@10"]),
            "Recall@10": float(row["Recall@10"]),
            "MRR": float(row["MRR"]),
            "NDCG@3": float(row["NDCG@3"])
        })

comparison = pd.DataFrame(comparison_rows)

# Deduplicate by model, preferring Week5 where appropriate.
comparison = comparison.drop_duplicates(
    subset=["model", "stage"],
    keep="first"
)

bm25_ndcg = float(
    comparison.loc[comparison["model"] == "BM25", "NDCG@10"].iloc[0]
)
xgb_ndcg = float(
    comparison.loc[comparison["model"] == "XGBoost Pointwise", "NDCG@10"].iloc[0]
)
lm_ndcg = float(
    comparison.loc[comparison["model"] == "LambdaMART", "NDCG@10"].iloc[0]
)

comparison["vs_BM25_%"] = (
    (comparison["NDCG@10"] - bm25_ndcg) / bm25_ndcg * 100
)
comparison["vs_XGBoost_%"] = (
    (comparison["NDCG@10"] - xgb_ndcg) / xgb_ndcg * 100
)

comparison = comparison.sort_values(
    "NDCG@10", ascending=False
).reset_index(drop=True)

print(comparison.to_string(index=False))
print(
    f"\nLambdaMART vs BM25: "
    f"{((lm_ndcg-bm25_ndcg)/bm25_ndcg*100):+.3f}%"
)
print(
    f"LambdaMART vs XGBoost Pointwise: "
    f"{((lm_ndcg-xgb_ndcg)/xgb_ndcg*100):+.3f}%"
)

# ------------------------------------------------------------
# STEP 4 — Define feature groups
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 4/8 — Define ablation feature groups")
print("=" * 74)

feature_groups = {
    "Lexical": [
        "bm25_score",
        "tfidf_similarity",
        "query_title_overlap",
        "query_description_overlap",
        "exact_title_match",
        "query_in_title",
        "query_title_token_coverage",
        "bm25_score_qnorm",
        "tfidf_similarity_qnorm",
    ],
    "Semantic_SBERT": [
        "sbert_similarity",
        "sbert_available",
        "sbert_similarity_qnorm",
    ],
    "Multimodal_CLIP": [
        "clip_similarity",
        "clip_available",
        "clip_similarity_qnorm",
    ],
    "Metadata_Structural": [
        "price",
        "average_rating",
        "review_count",
        "log_price",
        "log_review_count",
        "log_review_count_w5",
        "price_missing",
        "price_missing_w5",
        "rating_missing_w5",
        "query_len",
        "title_len",
        "description_len",
        "brand_in_query",
        "category_token_match",
    ],
}

# Keep only features that actually exist in this V1 dataset.
for group_name in list(feature_groups.keys()):
    feature_groups[group_name] = [
        c for c in feature_groups[group_name] if c in all_features
    ]

covered = set(sum(feature_groups.values(), []))
unassigned = [c for c in all_features if c not in covered]

print("Feature groups:")
for name, cols in feature_groups.items():
    print(f"  {name:22s}: {len(cols):2d} features")
if unassigned:
    print("  Unassigned:", unassigned)

group_rows = []
for group, cols in feature_groups.items():
    for c in cols:
        group_rows.append({"group": group, "feature": c})
for c in unassigned:
    group_rows.append({"group": "Unassigned", "feature": c})

group_map_df = pd.DataFrame(group_rows)

# ------------------------------------------------------------
# STEP 5 — Ablation study
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 5/8 — Train Full + feature-group ablation LambdaMART models")
print("=" * 74)

train_sorted = sort_rank(train_df)
test_sorted = sort_rank(test_df)

# Use exactly the selected V2 final hyperparameters.
final_params = dict(v2_config["final_hyperparameters"])

def train_and_eval(feature_subset, label):
    X_train = train_sorted[feature_subset]
    y_train = train_sorted["relevance_label"]
    X_test = test_sorted[feature_subset]

    model = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
        **final_params
    )

    model.fit(
        X_train,
        y_train,
        group=group_sizes(train_sorted)
    )

    temp = test_sorted[
        ["query_id", "product_id", "relevance_label"]
    ].copy()
    temp["score"] = model.predict(X_test)

    m = ranking_metrics(temp, "score")
    m["ablation"] = label
    m["num_features"] = len(feature_subset)
    return m, model

ablation_rows = []

# Full model reproducibility.
full_metrics, full_model = train_and_eval(
    all_features,
    "Full Model"
)
ablation_rows.append(full_metrics)

# Remove one group at a time.
for group_name, group_cols in feature_groups.items():
    remaining = [
        c for c in all_features if c not in set(group_cols)
    ]

    print(
        f"Training -{group_name}: "
        f"{len(remaining)} features remain"
    )

    m, _ = train_and_eval(
        remaining,
        f"Without {group_name}"
    )
    ablation_rows.append(m)

ablation = pd.DataFrame(ablation_rows)

full_ndcg = float(
    ablation.loc[
        ablation["ablation"] == "Full Model",
        "NDCG@10"
    ].iloc[0]
)

ablation["delta_NDCG@10"] = ablation["NDCG@10"] - full_ndcg
ablation["delta_NDCG@10_%"] = (
    (ablation["NDCG@10"] - full_ndcg)
    / full_ndcg
    * 100
)

ablation = ablation[
    [
        "ablation",
        "num_features",
        "NDCG@10",
        "NDCG@3",
        "Recall@10",
        "MRR",
        "delta_NDCG@10",
        "delta_NDCG@10_%",
        "Queries"
    ]
]

print("\nABLATION RESULTS")
print(ablation.to_string(index=False))

# ------------------------------------------------------------
# STEP 6 — Query-level LambdaMART vs XGBoost comparison
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 6/8 — Paired query-level XGBoost vs LambdaMART analysis")
print("=" * 74)

lm_scores = v2_rankings[
    [
        "query_id",
        "product_id",
        "relevance_label",
        "lambdamart_score"
    ]
].copy()

xgb_scores = xgb_pred[
    [
        "query_id",
        "product_id",
        "relevance_label",
        "xgb_score"
    ]
].copy()

paired = lm_scores.merge(
    xgb_scores,
    on=["query_id", "product_id"],
    how="inner",
    suffixes=("_lm", "_xgb")
)

if len(paired) != len(lm_scores):
    print(
        f"Warning: matched {len(paired):,} of "
        f"{len(lm_scores):,} LambdaMART test rows."
    )

query_rows = []

for qid, g in paired.groupby("query_id"):
    y = g["relevance_label_lm"].to_numpy(dtype=float)
    lm_s = g["lambdamart_score"].to_numpy(dtype=float)
    xgb_s = g["xgb_score"].to_numpy(dtype=float)

    lm_q = ndcg_score(
        y.reshape(1,-1),
        lm_s.reshape(1,-1),
        k=10
    )
    xgb_q = ndcg_score(
        y.reshape(1,-1),
        xgb_s.reshape(1,-1),
        k=10
    )

    query_rows.append({
        "query_id": qid,
        "LambdaMART_NDCG@10": float(lm_q),
        "XGBoost_NDCG@10": float(xgb_q),
        "difference": float(lm_q - xgb_q)
    })

query_compare = pd.DataFrame(query_rows)

EPS = 1e-12
wins = int((query_compare["difference"] > EPS).sum())
losses = int((query_compare["difference"] < -EPS).sum())
ties = int((query_compare["difference"].abs() <= EPS).sum())

# Paired bootstrap on query-level differences.
diffs = query_compare["difference"].to_numpy()
rng = np.random.RandomState(SEED)

B = 5000
boot_means = np.empty(B, dtype=float)

for i in range(B):
    sample = rng.choice(diffs, size=len(diffs), replace=True)
    boot_means[i] = sample.mean()

ci_low = float(np.percentile(boot_means, 2.5))
ci_high = float(np.percentile(boot_means, 97.5))
mean_diff = float(diffs.mean())
prob_positive = float((boot_means > 0).mean())

bootstrap_summary = pd.DataFrame([{
    "queries": len(query_compare),
    "LambdaMART_mean_NDCG@10_matched": query_compare["LambdaMART_NDCG@10"].mean(),
    "XGBoost_mean_NDCG@10_matched": query_compare["XGBoost_NDCG@10"].mean(),
    "mean_difference": mean_diff,
    "bootstrap_95%_CI_low": ci_low,
    "bootstrap_95%_CI_high": ci_high,
    "bootstrap_probability_difference_gt_0": prob_positive,
    "LambdaMART_query_wins": wins,
    "ties": ties,
    "XGBoost_query_wins": losses
}])

print(bootstrap_summary.to_string(index=False))

# ------------------------------------------------------------
# STEP 7 — Save tables + plots
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 7/8 — Save V3 outputs")
print("=" * 74)

OUT = ROOT / "output"
OUT.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    OUT / "week5_v3_model_comparison.csv",
    index=False
)

ablation.to_csv(
    OUT / "week5_v3_ablation_results.csv",
    index=False
)

group_map_df.to_csv(
    OUT / "week5_v3_ablation_feature_groups.csv",
    index=False
)

query_compare.to_csv(
    OUT / "week5_v3_querywise_xgb_vs_lambdamart.csv",
    index=False
)

bootstrap_summary.to_csv(
    OUT / "week5_v3_bootstrap_summary.csv",
    index=False
)

# Plot 1 — formal model comparison
plot_cmp = comparison.sort_values(
    "NDCG@10",
    ascending=True
)

plt.figure(figsize=(9, 6))
plt.barh(
    plot_cmp["model"] + " (" + plot_cmp["stage"] + ")",
    plot_cmp["NDCG@10"]
)
plt.xlabel("NDCG@10")
plt.ylabel("Model")
plt.title("Week 5 V3 — Ranking Model Comparison")
plt.xlim(
    max(0, plot_cmp["NDCG@10"].min() - 0.05),
    min(1.0, plot_cmp["NDCG@10"].max() + 0.03)
)
plt.tight_layout()
plt.savefig(
    OUT / "week5_v3_model_comparison.png",
    dpi=180,
    bbox_inches="tight"
)
plt.close()

# Plot 2 — ablation
plot_abl = ablation.sort_values(
    "NDCG@10",
    ascending=True
)

plt.figure(figsize=(10, 6))
plt.barh(
    plot_abl["ablation"],
    plot_abl["NDCG@10"]
)
plt.xlabel("NDCG@10")
plt.ylabel("Model Variant")
plt.title("Week 5 V3 — LambdaMART Feature-Group Ablation")
plt.xlim(
    max(0, plot_abl["NDCG@10"].min() - 0.03),
    min(1.0, plot_abl["NDCG@10"].max() + 0.02)
)
plt.tight_layout()
plt.savefig(
    OUT / "week5_v3_ablation.png",
    dpi=180,
    bbox_inches="tight"
)
plt.close()

# Plot 3 — query win/tie/loss
wtl = pd.DataFrame({
    "Outcome": [
        "LambdaMART wins",
        "Ties",
        "XGBoost wins"
    ],
    "Queries": [wins, ties, losses]
})

plt.figure(figsize=(7, 5))
plt.bar(
    wtl["Outcome"],
    wtl["Queries"]
)
plt.ylabel("Number of Test Queries")
plt.title("Week 5 V3 — Query-Level NDCG@10 Win / Tie / Loss")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(
    OUT / "week5_v3_query_wins.png",
    dpi=180,
    bbox_inches="tight"
)
plt.close()

# Determine largest harmful ablation (largest drop).
removed = ablation[
    ablation["ablation"] != "Full Model"
].copy()

most_important_row = removed.sort_values(
    "delta_NDCG@10"
).iloc[0]

lm_vs_bm25 = (lm_ndcg - bm25_ndcg) / bm25_ndcg * 100
lm_vs_xgb = (lm_ndcg - xgb_ndcg) / xgb_ndcg * 100

readme = f"""
AMAZON SEARCH / LTR — WEEK 5 V3
MODEL COMPARISON + FEATURE-GROUP ABLATION
=========================================

FORMAL TEST COMPARISON
----------------------
BM25 NDCG@10:              {bm25_ndcg:.6f}
Week2 XGBoost NDCG@10:     {xgb_ndcg:.6f}
Week5 LambdaMART NDCG@10:  {lm_ndcg:.6f}

LambdaMART vs BM25:
{lm_vs_bm25:+.3f}%

LambdaMART vs Week2 XGBoost:
{lm_vs_xgb:+.3f}%

This means LambdaMART exceeds the Week2 pointwise model, but not by 12%.
The original roadmap's "+12% over Week2 pointwise" target is impossible here
because 0.922169 * 1.12 > 1.0, while NDCG cannot exceed 1.0.

LambdaMART DOES exceed BM25 by more than 12%, which is a meaningful and
mathematically valid retrieval-to-LTR improvement.

ABLATION DESIGN
---------------
The final V2 hyperparameters are held fixed.
One feature group is removed at a time.
The model is retrained on all 800 training queries and evaluated on the same
untouched 200-query test set.

Largest NDCG@10 drop:
{most_important_row['ablation']}
Delta: {most_important_row['delta_NDCG@10']:+.6f}
Relative delta: {most_important_row['delta_NDCG@10_%']:+.3f}%

PAIRED QUERY ANALYSIS
---------------------
Matched test queries: {len(query_compare)}
LambdaMART query wins: {wins}
Ties: {ties}
XGBoost query wins: {losses}

Mean paired NDCG@10 difference:
{mean_diff:+.6f}

Paired bootstrap 95% CI:
[{ci_low:+.6f}, {ci_high:+.6f}]

Bootstrap probability that LambdaMART mean difference > 0:
{prob_positive:.4f}

INTERPRETATION NOTE
-------------------
Use the paired bootstrap CI to describe whether the small LambdaMART-vs-XGBoost
advantage appears stable across test queries. Avoid claiming statistical
significance if the interval includes zero.

NEXT
----
Week5 V4:
- final validation / consolidated Week5 tables
- optional ListNet only if required by the roadmap
- final Week5 technical report
""".strip()

with open(
    OUT / "README_WEEK5_V3.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(readme + "\n")

# ------------------------------------------------------------
# STEP 8 — ZIP + download
# ------------------------------------------------------------
print("\n" + "=" * 74)
print("STEP 8/8 — Package V3 results")
print("=" * 74)

zip_out = Path(
    "/content/amazon_week5_v3_model_comparison_ablation_results.zip"
)

if zip_out.exists():
    zip_out.unlink()

with zipfile.ZipFile(
    zip_out,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.name)

print("\n✅ WEEK 5 V3 COMPLETE")
print("✅ Created:", zip_out)
print("\nUpload the V3 ZIP back to ChatGPT for the next step.")
files.download(str(zip_out))
